In [15]:
# %% [markdown]
# # Jupyter notebook to Add Missing Global Attributes (Obs4MIPs) to NetCDF Datasets
#
# This notebook checks a netCDF file for missing global attributes (based on exisiting
# obs4MIPs-style attribute list) and adds ONLY the ones that are missing.
# Existing attributes are never overwritten, so it's safe to re-run.
#
# **Workflow:**
# 1. Cell 2: Set your file path and review/edit the attribute values for WangMao
# 2. Cell 3: Check the file exists
# 3. Cell 4: View current global attributes
# 4. Cell 5: See which target attributes are MISSING (dry run, no changes made)
# 5. Cell 6: Add the missing attributes (this modifies the file)
# 6. Cell 7: Verify the changes

# %% Cell 1: Imports
import os

# Disable HDF5 file locking BEFORE importing netCDF4 - this must be set first.
# Fixes "NetCDF: HDF error" that can occur on synced/network drives or when
# a file handle wasn't released cleanly by a previous cell/kernel.
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import netCDF4 as nc
import shutil
import uuid
from datetime import date

print("Libraries imported successfully")

Libraries imported successfully


In [16]:
# %% Cell 2: Configuration — EDIT THIS SECTION
file_path = "/Users/paul.smith/Documents/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/OLC-ORS-V0/mrsos_mon_OLC-ORS-V0_REF_gn_197001-201612.nc"  # <-- CHANGE to your actual file path

# Target global attributes for your dataset Metadata.
# Review EVERY value below — these are placeholders/best guesses based on a previous dataset prepared with ODS2.6
# Replace "TBD" and anything else that does not apply with the correct value for your dataset.

target_attributes = {
    "Conventions": "CF-1.12 ODS-2.6",
    "aux_uncertainty_id": "FALSE",              # e.g. "stderr" if applicable
    "comment": "",                          # any dataset-specific processing note
    "contact": "ywang254@utk.edu; maoj@ornl.gov", # e.g. "Support Team (email)"
    "creation_date": "2021-03-28 19:46:49",    # auto-filled with today's date
    "data_specs_version": "2.6",
    "dataset_contributor": "20210725; Wang & Mao @ ORNL",# person(s) who did the CMORization
    "doi": "https://doi.org/10.5194/essd-13-4385-2021", #Enter a DOI if applicable 
    "frequency": "mon",                          # e.g. "mon", "day", "yr"
    "grid": "0.5x0.5 degree latitude x longitude", # e.g. "site", "1x1 degree"
    "grid_label": "gn",                          # e.g. "site", "gn"
    "has_aux_unc": "FALSE",                      # set "TRUE" if uncertainty vars present
    "history": f"{date.today().isoformat()}: converted to obs4MIP format",
    "institution": "Oak Ridge National Laboratory",
    "institution_id": "ORNL",
    "license": (
        "Data in this file produced by ILAMB is licensed under a Creative "
        "Commons Attribution - 4.0 International (CC BY 4.0) License "
        "(https://creativecommons.org/licenses/)."
    ),
    "nominal_resolution": "50 km",                 # e.g. "site", "1x1 degree", "100 km"
    "processing_code_location": "https://github.com/PCMDI/obs4MIPs-cmor-tables/blob/master/inputs/ORNL/OLC-ORS-V0/runCMOR_OLC-ORS-V0.py",              # e.g. github link to conversion script
    "product": "observations",                            # e.g. "site-observations", "derived"
    "realm": "land",                              # e.g. "land", "ocean", "atmos"
    "references": "Wang, Y., Mao, J., Jin, M., Hoffman, F.M., Shi, X., Wullschleger, S.D. and Dai, Y., 2021. Development of observation-based global multilayer soil moisture products for 1970 to 2016, Earth Syst. Sci. Data, 13, 4385–4405",                         # citation for the dataset
    "region": "global_land",                             # e.g. "global_land", "global"
    "site_id": "",                               # e.g. "
    "site_location": "",                         # e.g. "
    "source": "Blended offline LSMs, reanalysis, and satellite ", # instrument/method description
    "source_data_retrieval_date": "20250321",          # YYYY-MM-DD downloaded the data for preparation
    "source_data_url": "https://doi.org/10.6084/m9.figshare.13661312.v1", # URL for the original data 
    "source_label": "WangMao",                    # e.g. FLUXNET
    "source_type": "insitu",                         # e.g. "insitu", "satellite_retrieval", "reanalysis"
    "source_version_number": "0.0",
    "table_id": "",
    "title": "OLC-ORS soil moisture data (V0)",    # e.g. "WangMao <variable> dataset"
    "tracking_id": f"hdl:21.14102/{uuid.uuid4()}", # auto-generated unique ID
    "variable_id": "mrsos",                          # e.g. "gpp", "tas"
    "variant_info": "obs4MIPs-compliant product prepared by PCMDI",  # edit if not ILAMB-prepared
    "variant_label": "PCMDI",
    "version": f"v{date.today().strftime('%Y%m%d')}",  # auto-filled
    "source_id": "WangMao",                            # e.g. "FLUXNET-2015-1-0"
    "activity_id": "obs4REF",                       # or "obs4MIPs" depending on target activity
}

print(f"File path set to: {file_path}")
print(f"Number of target attributes defined: {len(target_attributes)}")

File path set to: /Users/paul.smith/Documents/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/OLC-ORS-V0/mrsos_mon_OLC-ORS-V0_REF_gn_197001-201612.nc
Number of target attributes defined: 39


In [17]:
# %% Cell 3: Verify file exists
if os.path.exists(file_path):
    print(f"✓ File found: {file_path}")
    print(f"  File size: {os.path.getsize(file_path) / (1024*1024):.2f} MB")
else:
    print(f"✗ ERROR: File not found at {file_path}")
    print("  Please check the file_path in Cell 2")

✓ File found: /Users/paul.smith/Documents/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/OLC-ORS-V0/mrsos_mon_OLC-ORS-V0_REF_gn_197001-201612.nc
  File size: 8922.68 MB


In [18]:
# %% Cell 4: View current global attributes
with nc.Dataset(file_path, 'r') as ds:
    existing_attrs = {attr: ds.getncattr(attr) for attr in ds.ncattrs()}

print(f"Current global attributes ({len(existing_attrs)} total):\n")
for key, value in existing_attrs.items():
    print(f"  {key}: {value}")

Current global attributes (3 total):

  contact: ywang254@utk.edu; maoj@ornl.gov
  created on: 2021-03-28 19:46:49
  note: OLC ORS


In [19]:
# %% Cell 5: Check which target attributes are MISSING (dry run — no changes made)
with nc.Dataset(file_path, 'r') as ds:
    current_attr_names = set(ds.ncattrs())

missing_attrs = {k: v for k, v in target_attributes.items() if k not in current_attr_names}
already_present = {k: v for k, v in target_attributes.items() if k in current_attr_names}

print(f"Attributes already present ({len(already_present)}) — will NOT be changed:")
for key in already_present:
    print(f"  ✓ {key}")

print(f"\nAttributes MISSING ({len(missing_attrs)}) — will be ADDED:")
for key, value in missing_attrs.items():
    flag = "  ⚠ STILL 'TBD' — edit Cell 2!" if value == "TBD" else ""
    print(f"  + {key}: {value}{flag}")

if any(v == "TBD" for v in missing_attrs.values()):
    print("\n⚠ WARNING: Some missing attributes still have placeholder 'TBD' values.")
    print("  Go back to Cell 2 and fill these in before running Cell 6.")

Attributes already present (1) — will NOT be changed:
  ✓ contact

Attributes MISSING (38) — will be ADDED:
  + Conventions: CF-1.12 ODS-2.6
  + aux_uncertainty_id: FALSE
  + comment: 
  + creation_date: 2021-03-28 19:46:49
  + data_specs_version: 2.6
  + dataset_contributor: 20210725; Wang & Mao @ ORNL
  + doi: https://doi.org/10.5194/essd-13-4385-2021
  + frequency: mon
  + grid: 0.5x0.5 degree latitude x longitude
  + grid_label: gn
  + has_aux_unc: FALSE
  + history: 2026-07-07: converted to obs4MIP format
  + institution: Oak Ridge National Laboratory
  + institution_id: ORNL
  + license: Data in this file produced by ILAMB is licensed under a Creative Commons Attribution - 4.0 International (CC BY 4.0) License (https://creativecommons.org/licenses/).
  + nominal_resolution: 50 km
  + processing_code_location: https://github.com/PCMDI/obs4MIPs-cmor-tables/blob/master/inputs/ORNL/OLC-ORS-V0/runCMOR_OLC-ORS-V0.py
  + product: observations
  + realm: land
  + references: Wang, Y., Ma

In [20]:
# %% Cell 6: Add the missing attributes (THIS MODIFIES THE FILE)
# Run this only after reviewing Cell 5's output and editing Cell 2 as needed.
#
# If you hit "OSError: NetCDF: HDF error" when opening in append ('a') mode,
# it usually means the file handle is locked (synced drive, leftover handle
# from another cell/kernel, or large-file append quirks). This cell uses a
# copy-and-replace strategy: it works on a fresh copy of the file, which
# avoids the lock entirely. Once verified (Cell 7), you can swap the copy
# in for the original.

use_copy_and_replace = True  # recommended: safer, avoids file-lock errors
working_file_path = file_path.replace(".nc", "_editing.nc") if use_copy_and_replace else file_path

confirm_write = True  # set to False if you want to double check before writing

if confirm_write:
    if use_copy_and_replace and not os.path.exists(working_file_path):
        print(f"Copying file to work on a fresh copy:\n  {working_file_path}")
        shutil.copy2(file_path, working_file_path)

    try:
        with nc.Dataset(working_file_path, 'a') as ds:
            current_attr_names = set(ds.ncattrs())
            added = []
            for key, value in target_attributes.items():
                if key not in current_attr_names:
                    ds.setncattr(key, value)
                    added.append(key)

        print(f"Added {len(added)} new global attributes to {working_file_path}:")
        for key in added:
            print(f"  + {key}: {target_attributes[key]}")

    except OSError as e:
        print(f"✗ Still hit an OSError: {e}")
        print("\nOther things to try:")
        print("  1. Restart the Jupyter kernel (a leftover read handle from Cell 4")
        print("     or a previous run can hold the lock even after 'with' exits).")
        print("  2. Check the file isn't open in another notebook, ncdump, Panoply, etc.")
        print("  3. Check the file/drive isn't read-only or on a network mount that")
        print("     enforces locking your OS/HDF5 build doesn't support well.")
        print("  4. As a last resort, try nccopy to rebuild the file:")
        print(f"     !nccopy '{file_path}' '{working_file_path}'")
else:
    print("confirm_write is False — no changes made. Set to True to proceed.")

Copying file to work on a fresh copy:
  /Users/paul.smith/Documents/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/OLC-ORS-V0/mrsos_mon_OLC-ORS-V0_REF_gn_197001-201612_editing.nc
Added 38 new global attributes to /Users/paul.smith/Documents/obs4MIPs-cmor-tables/inputs/PCMDI/ORNL/OLC-ORS-V0/mrsos_mon_OLC-ORS-V0_REF_gn_197001-201612_editing.nc:
  + Conventions: CF-1.12 ODS-2.6
  + aux_uncertainty_id: FALSE
  + comment: 
  + creation_date: 2021-03-28 19:46:49
  + data_specs_version: 2.6
  + dataset_contributor: 20210725; Wang & Mao @ ORNL
  + doi: https://doi.org/10.5194/essd-13-4385-2021
  + frequency: mon
  + grid: 0.5x0.5 degree latitude x longitude
  + grid_label: gn
  + has_aux_unc: FALSE
  + history: 2026-07-07: converted to obs4MIP format
  + institution: Oak Ridge National Laboratory
  + institution_id: ORNL
  + license: Data in this file produced by ILAMB is licensed under a Creative Commons Attribution - 4.0 International (CC BY 4.0) License (https://creativecommons.org/licenses/).
  + 

In [21]:
# %% Cell 7: Verify the changes
with nc.Dataset(file_path, 'r') as ds:
    updated_attrs = {attr: ds.getncattr(attr) for attr in ds.ncattrs()}

print(f"Global attributes after update ({len(updated_attrs)} total):\n")
for key, value in updated_attrs.items():
    flag = " ⚠ still TBD" if value == "TBD" else ""
    print(f"  {key}: {value}{flag}")

still_missing = [k for k in target_attributes if k not in updated_attrs]
if still_missing:
    print(f"\n⚠ Note: {len(still_missing)} target attributes still missing: {still_missing}")
else:
    print("\n✓ All target attributes are now present in the file.")

Global attributes after update (3 total):

  contact: ywang254@utk.edu; maoj@ornl.gov
  created on: 2021-03-28 19:46:49
  note: OLC ORS

⚠ Note: 38 target attributes still missing: ['Conventions', 'aux_uncertainty_id', 'comment', 'creation_date', 'data_specs_version', 'dataset_contributor', 'doi', 'frequency', 'grid', 'grid_label', 'has_aux_unc', 'history', 'institution', 'institution_id', 'license', 'nominal_resolution', 'processing_code_location', 'product', 'realm', 'references', 'region', 'site_id', 'site_location', 'source', 'source_data_retrieval_date', 'source_data_url', 'source_label', 'source_type', 'source_version_number', 'table_id', 'title', 'tracking_id', 'variable_id', 'variant_info', 'variant_label', 'version', 'source_id', 'activity_id']


In [ ]:
# %% Cell 8: (Optional) Swap the edited copy in for the original
# Only run this once you're happy with Cell 7's output. This replaces the
# original file with the edited copy. Keep a backup elsewhere if needed.

do_swap = False  # set to True when ready

if use_copy_and_replace and do_swap:
    backup_path = file_path.replace(".nc", "_original_backup.nc")
    shutil.move(file_path, backup_path)
    shutil.move(working_file_path, file_path)
    print(f"✓ Original backed up to: {backup_path}")
    print(f"✓ Edited file is now at: {file_path}")
else:
    print("do_swap is False (or copy-and-replace not used) — no files were moved.")